In [1]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

%cd /content
!git clone https://{GITHUB_TOKEN}@github.com/Bassey-data/Auto-insurance-claim-frequency.git

%cd /content/Auto-insurance-claim-frequency
!git config user.email "basseysamuel404@gmail.com"
!git config user.name "Bassey-data"

print("Setup complete")

/content
Cloning into 'Auto-insurance-claim-frequency'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 21 (delta 5), reused 15 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 63.04 KiB | 5.73 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/Auto-insurance-claim-frequency
Setup complete


In [2]:
pip install statsmodels liac-arff lightgbm pyarrow shap

  Preparing metadata (setup.py) ... done
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11717 sha256=70ac818aa7899382234ed600f1d43900ac4ebf819b5f76e4c211baa47f953041
  Stored in directory: /root/.cache/pip/wheels/a9/ac/cf/c2919807a5c623926d217c0a18eb5b457e5c19d242c3b5963a
Successfully built liac-arff


Import all useful modules

In [ ]:

import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_poisson_deviance
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import warnings
warnings.filterwarnings("ignore")


In [ ]:
# Plot styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

In [ ]:
def parse_arff(path):
    columns = []
    data_start = None

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped.lower().startswith("@attribute"):
            parts = stripped.split(maxsplit=2)
            columns.append(parts[1])
        elif stripped.lower().startswith("@data"):
            data_start = i + 1
            break

    data_lines = [l.strip() for l in lines[data_start:] if l.strip()]

    rows = []
    for line in data_lines:
        values = [v.strip().strip("'").strip('"') for v in line.split(",")]
        rows.append(values)

    return pd.DataFrame(rows, columns=columns)


df = parse_arff("/content/drive/MyDrive/ACQsci.arff")

print(df.head())

In [ ]:
df.dtypes

fix numeric and categorical columns data types

In [ ]:
# used an f string for readable output
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isna().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

In [ ]:
df_converted = df.apply(pd.to_numeric, errors="ignore")

for col in df_converted.select_dtypes(include="object").columns:
    df_converted[col] = df_converted[col].astype("category")

df = df_converted
print(df.dtypes)

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isna().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

In [ ]:
os.makedirs("data/processed", exist_ok=True)

df.to_parquet("data/processed/freMTPL2freq.parquet", index=False)

print("Saved successfully")
print(f"Parquet size: {os.path.getsize('data/processed/freMTPL2freq.parquet') / 1e6:.1f} MB")

In [ ]:
print(os.path.exists("/content/Auto-insurance-claim-frequency/data/processed/freMTPL2freq.parquet"))


In [ ]:
%cd /content/Auto-insurance-claim-frequency
!git add .
!git commit -m "Add data preparation notebook and processed parquet file"
!git push

In [ ]:
%cd /content/Auto-insurance-claim-frequency
!ls -la
!ls -la data/processed/
!ls -la notebooks/ sce

In [ ]:
%cd /content/Auto-insurance-claim-frequency

# Create notebooks folder and copy notebook into it
!mkdir -p notebooks
!cp "/content/drive/MyDrive/Colab Notebooks/Insurance_claim.ipynb" notebooks/01_data_prep.ipynb

# The parquet is gitignored which is correct - data files shouldn't be on GitHub
# Let's confirm notebooks got copied
!ls notebooks/

In [ ]:
!git add notebooks/
!git commit -m "Add data preparation notebook"
!git push

In [ ]:
df = pd.read_parquet("/content/Auto-insurance-claim-frequency/data/processed/freMTPL2freq.parquet")
print(df.shape)
print(df.head())

Check total clims and total exposure and calculate portfolio frequency

In [ ]:
total_claims = df["ClaimNb"].sum()
total_exposure = df["Exposure"].sum()
portfolio_frequency = total_claims / total_exposure

print(f"Total claims: {total_claims:,}")
print(f"Total exposure (policy-years): {total_exposure:,.1f}")
print(f"Portfolio frequency: {portfolio_frequency:.4f} claims per policy-year")

Check claim distribution

In [ ]:
claim_counts = df["ClaimNb"].value_counts().sort_index()
print(claim_counts)
print()
print(f"Policies with zero claims: {(df['ClaimNb'] == 0).mean() * 100:.2f}%")

Visualize using log scale

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(claim_counts.index.astype(str), claim_counts.values, color="steelblue")
ax.set_yscale("log")
ax.set_title("Distribution of ClaimNb (log scale)")
ax.set_xlabel("Number of claims")
ax.set_ylabel("Number of policies (log scale)")

plt.tight_layout()
plt.show()

Exposure plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(df["Exposure"], bins=50, color="darkorange")
ax.set_title("Distribution of Exposure")
ax.set_xlabel("Exposure (fraction of year)")
ax.set_ylabel("Number of policies")

plt.tight_layout()
plt.show()

print(f"Min exposure: {df['Exposure'].min()}")
print(f"Max exposure: {df['Exposure'].max()}")
print(f"Policies with Exposure > 1: {(df['Exposure'] > 1).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["DrivAge"], bins=50, color="seagreen")
axes[0].set_title("Distribution of DrivAge")
axes[0].set_xlabel("Driver age")

axes[1].hist(df["VehAge"], bins=50, color="indianred")
axes[1].set_title("Distribution of VehAge")
axes[1].set_xlabel("Vehicle age (years)")

plt.tight_layout()
plt.show()

print(f"DrivAge range: {df['DrivAge'].min()} - {df['DrivAge'].max()}")
print(f"VehAge range: {df['VehAge'].min()} - {df['VehAge'].max()}")
print(f"Policies with VehAge > 30: {(df['VehAge'] > 30).sum()}")

In [ ]:
def exposure_weighted_frequency(df, group_col):
    # Group by the column and sum claims and exposure separately
    g = df.groupby(group_col, observed=True).agg(
        claims=("ClaimNb", "sum"),    # total claims in this group
        exposure=("Exposure", "sum")  # total exposure in this group
    )

    # Divide total claims by total exposure to get the true rate
    # This is correct - NOT the mean of (ClaimNb/Exposure) per policy
    g["frequency"] = g["claims"] / g["exposure"]

    return g.sort_index()

In [ ]:
# Plot frequency by each categorical feature
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

cat_cols = ["Area", "VehGas", "Region", "VehBrand"]

for ax, col in zip(axes.flat, cat_cols):
    # Get exposure-weighted frequency for this column
    freq_table = exposure_weighted_frequency(df, col)

    # Plot as bar chart
    ax.bar(freq_table.index.astype(str), freq_table["frequency"], color="slateblue")

    # Add a red dashed line at the portfolio average so we can see above/below average groups
    ax.axhline(portfolio_frequency, color="red", linestyle="--", linewidth=1, label="Portfolio avg")

    ax.set_title(f"Claim frequency by {col}")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bin DrivAge into 20 groups and plot frequency per bin
df["DrivAge_bin"] = pd.cut(df["DrivAge"], bins=20)
freq_age = exposure_weighted_frequency(df, "DrivAge_bin")
x_labels = [f"{i.left:.0f}" for i in freq_age.index]
axes[0].plot(x_labels, freq_age["frequency"], marker="o", color="crimson")
axes[0].set_title("Claim frequency by DrivAge (binned)")
axes[0].tick_params(axis="x", rotation=60)
axes[0].axhline(portfolio_frequency, color="blue", linestyle="--", linewidth=1)

# Same for BonusMalus
df["BonusMalus_bin"] = pd.cut(df["BonusMalus"], bins=20)
freq_bm = exposure_weighted_frequency(df, "BonusMalus_bin")
x_labels_bm = [f"{i.left:.0f}" for i in freq_bm.index]
axes[1].plot(x_labels_bm, freq_bm["frequency"], marker="o", color="darkorange")
axes[1].set_title("Claim frequency by BonusMalus (binned)")
axes[1].tick_params(axis="x", rotation=60)
axes[1].axhline(portfolio_frequency, color="blue", linestyle="--", linewidth=1)

plt.tight_layout()
plt.show()